# AIaaS — Model-Swap / Cold-Start Benchmark

The single A100 (40 GB) can't hold every workload resident at once. This notebook
measures, per workload, the cost of **bringing a model in and out of the GPU**:

- **load time** (cold, disk/CPU → GPU) and **unload time**
- **cold first-inference** vs **warm inference** (the kernel/compile warmup tax)
- **resident VRAM** (steady-state) and **peak VRAM** (during inference)

Then it answers the capacity question directly: **do the workloads co-reside in
this GPU, and if not, what does a swap cost?**

### Why this matters
Throughput benchmarks assume the model is already loaded. A multi-tenant AIaaS box
that serves LLM + embeddings + image-gen from one 40 GB card may have to swap
models on demand — and the swap latency (unload + load + cold first call) is a
real tail-latency and capacity cost the other notebooks don't capture.

### Requirements
- A **CUDA GPU**. VRAM-tiered, ungated models. Runs on a T4 (smaller models) or
  an A100 (larger). No accuracy claims — this is a *systems* measurement.

## 1. Install

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "sentence-transformers", "diffusers",
                "accelerate", "pandas"], check=False)
import transformers, diffusers
print("transformers", transformers.__version__, "| diffusers", diffusers.__version__)


## 2. Environment & hardware capture

In [ ]:
import os, json, platform, subprocess, datetime
import torch

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

CUDA = torch.cuda.is_available()
assert CUDA, "No CUDA GPU detected. This benchmark requires a GPU runtime."
_cc = torch.cuda.get_device_capability(0)

ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "compute_capability": f"{_cc[0]}.{_cc[1]}",
    "cuda": torch.version.cuda,
    "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__,
    "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))


## 3. Configuration
VRAM-tiered, ungated models for the three resident workloads. Edit `WORKLOADS` to
match what your platform actually serves.

In [ ]:
VRAM = ENV["vram_total_gb"]
TIER = "large" if VRAM >= 24 else "small"
DTYPE = "bfloat16" if _cc[0] >= 8 else "float16"

if TIER == "large":
    LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
    IMG_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
    IMG_STEPS, IMG_SIZE = 20, 1024
else:
    LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    IMG_MODEL = "stabilityai/sd-turbo"
    IMG_STEPS, IMG_SIZE = 2, 512

EMB_MODEL = "BAAI/bge-small-en-v1.5"

WORKLOADS = [
    {"name": "llm",        "kind": "llm", "model": LLM_MODEL},
    {"name": "embeddings", "kind": "emb", "model": EMB_MODEL},
    {"name": "image_gen",  "kind": "img", "model": IMG_MODEL},
]

OUT_DIR = "model_swap_results"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"VRAM={VRAM} GB -> TIER={TIER}, dtype={DTYPE}")
for w in WORKLOADS:
    print(f"  {w['name']:11s} -> {w['model']}")


## 4. Loaders + the measurement harness
Each loader returns the live objects plus an `infer()` closure. The harness times
load, cold/warm inference, and unload, and records resident/peak VRAM.

In [ ]:
import time, gc

dtype = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16

def load_llm(model_id):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_id)
    m = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype).to("cuda")
    ids = tok("Summarize: the quick brown fox jumps over the lazy dog.",
              return_tensors="pt").to("cuda")
    def infer():
        with torch.no_grad():
            m.generate(**ids, max_new_tokens=16, do_sample=False)
    return {"objs": [m, tok, ids], "infer": infer}

def load_emb(model_id):
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(model_id, device="cuda")
    if dtype in (torch.float16, torch.bfloat16):   # match the tier dtype (bf16 on A100)
        m = m.to(dtype)
    def infer():
        m.encode(["hello world"] * 16, show_progress_bar=False, convert_to_numpy=True)
    return {"objs": [m], "infer": infer}

def load_img(model_id):
    from diffusers import AutoPipelineForText2Image
    p = AutoPipelineForText2Image.from_pretrained(model_id, torch_dtype=dtype).to("cuda")
    p.set_progress_bar_config(disable=True)
    def infer():
        p(prompt="a cat", num_inference_steps=IMG_STEPS, height=IMG_SIZE, width=IMG_SIZE)
    return {"objs": [p], "infer": infer}

LOADERS = {"llm": load_llm, "emb": load_emb, "img": load_img}

def measure(workload):
    kind, model_id = workload["kind"], workload["model"]
    gc.collect(); torch.cuda.empty_cache()
    base = torch.cuda.memory_allocated()

    t = time.time()
    state = LOADERS[kind](model_id)
    torch.cuda.synchronize(); load_s = time.time() - t
    resident_gb = (torch.cuda.memory_allocated() - base) / 1e9

    # Isolate the INFERENCE peak: reset the high-water mark AFTER load so the
    # peak reflects inference (resident + activations), not the load transient.
    torch.cuda.reset_peak_memory_stats()
    t = time.time(); state["infer"](); torch.cuda.synchronize(); cold_s = time.time() - t
    t = time.time(); state["infer"](); torch.cuda.synchronize(); warm_s = time.time() - t
    peak_gb = (torch.cuda.max_memory_allocated() - base) / 1e9

    t = time.time()
    state["objs"].clear(); del state
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    unload_s = time.time() - t

    # Verify the GPU actually returned to baseline; un-reclaimed memory would
    # inflate the next workload's base and under-count its resident footprint.
    residual_gb = (torch.cuda.memory_allocated() - base) / 1e9
    if residual_gb > 0.1:
        print(f"  WARNING: {workload['name']} left {round(residual_gb, 2)} GB allocated after "
              "unload (empty_cache couldn't reclaim it) — later residents may be under-counted.")

    return {
        "name": workload["name"], "model": model_id,
        "load_s": round(load_s, 3),
        "cold_infer_s": round(cold_s, 3),
        "warm_infer_s": round(warm_s, 3),
        "cold_start_tax_s": round(cold_s - warm_s, 3),
        "resident_vram_gb": round(resident_gb, 2),
        "peak_vram_gb": round(peak_gb, 2),
        "unload_s": round(unload_s, 3),
        "swap_in_out_s": round(load_s + unload_s, 3),
        "post_unload_residual_gb": round(residual_gb, 3),
    }


## 5. Run — one workload at a time (load → infer → unload)

In [ ]:
RESULTS = []
for w in WORKLOADS:
    print(f"\n=== {w['name']} ({w['model']}) ===")
    try:
        r = measure(w)
    except Exception as e:
        r = {"name": w["name"], "model": w["model"], "error": f"{type(e).__name__}: {e}"}
    print(r)
    RESULTS.append(r)


## 6. Results — save + summarize

In [ ]:
import pandas as pd
NORM = {
    "schema": "model-swap-bench/1.0",
    "env": ENV, "tier": TIER, "dtype": DTYPE,
    "results": RESULTS,
}
tag = (ENV["platform"] + "_" + ENV["gpu_name"]).replace(" ", "-").replace("/", "-")
out = os.path.join(OUT_DIR, f"model_swap_{tag}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)
print("Saved:", out)
print(f"\n==== MODEL-SWAP SUMMARY ====\n{ENV['gpu_name']} ({ENV['vram_total_gb']} GB) | {DTYPE}")
df = pd.DataFrame([r for r in RESULTS if "error" not in r])
try:
    from IPython.display import display; display(df)
except Exception:
    print(df.to_string(index=False))


## 7. Co-residency verdict & swap-cost analysis
Can all workloads stay resident? If not, how expensive is swapping?

In [ ]:
ok = [r for r in RESULTS if "error" not in r]
errored = [r for r in RESULTS if "error" in r]
usable = round(0.90 * ENV["vram_total_gb"], 1)   # leave ~10% headroom for activations
resident_total = round(sum(r["resident_vram_gb"] for r in ok), 2)
residual_total = round(sum(r.get("post_unload_residual_gb", 0.0) for r in ok), 2)

print(f"Usable VRAM (~90% of {ENV['vram_total_gb']} GB): {usable} GB")
print(f"Sum of resident footprints: {resident_total} GB "
      f"({', '.join(r['name'] + '=' + str(r['resident_vram_gb']) for r in ok)})")
if errored:
    print(f"EXCLUDED {len(errored)} workload(s) that failed to load/run: "
          f"{', '.join(r['name'] for r in errored)}")
if residual_total > 0.1:
    print(f"NOTE: {residual_total} GB was not reclaimed after unloads — resident_total "
          "may be under-counted.")

# A workload that couldn't even load cannot co-reside: never give a green verdict
# while any workload errored.
fits = (resident_total <= usable) and not errored
ANALYSIS = {
    "usable_vram_gb": usable, "resident_total_gb": resident_total,
    "measured_workloads": len(ok), "excluded_workloads": [r["name"] for r in errored],
    "unreclaimed_after_unload_gb": residual_total,
    "all_co_resident": fits,
}

if errored:
    print(f"\nVERDICT: INCONCLUSIVE — {len(errored)} workload(s) failed to load/run, so "
          "co-residency cannot be confirmed (a workload that won't load can't co-reside). "
          "Re-run on a larger GPU or with smaller models.")
elif fits:
    print("\nVERDICT: all workloads CO-RESIDE — no swapping needed. "
          f"Spare: {round(usable - resident_total, 2)} GB.")
else:
    print("\nVERDICT: workloads do NOT all fit — model-swap required on demand.")
    worst = max(ok, key=lambda r: r["swap_in_out_s"])
    print(f"Most expensive single swap-in+out: {worst['name']} "
          f"= {worst['swap_in_out_s']} s (load {worst['load_s']} + unload {worst['unload_s']})")
    print("Add the cold-start tax on the first call after each swap:")
    for r in ok:
        print(f"  {r['name']:11s} load {r['load_s']}s + cold-start {r['cold_start_tax_s']}s")
    ANALYSIS["worst_swap_in_out_s"] = worst["swap_in_out_s"]
    ANALYSIS["per_workload_swap"] = {
        r["name"]: {"load_s": r["load_s"], "unload_s": r["unload_s"],
                    "cold_start_tax_s": r["cold_start_tax_s"]} for r in ok}

# persist the verdict alongside the raw results
NORM["analysis"] = ANALYSIS
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)
print("\nUpdated", out)


## 8. Reading the result

- **resident_vram_gb** is what each model costs to keep loaded; their sum vs.
  usable VRAM is the co-residency test.
- If they don't co-reside, **swap_in_out_s** (+ the **cold-start tax** on the
  first call) is the latency a request pays when its model isn't already loaded —
  feed it into your capacity/SLA model and your keep-resident vs. load-on-demand
  policy.
- Pairs naturally with the serving numbers: a workload that swaps in slowly but
  serves fast may still be worth keeping warm.

This is the systems-level companion to the throughput notebooks; see
`BENCHMARKING_STRATEGY.md` (the 40 GB co-residency constraint).